In [12]:
import config
import data_logging

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1.数据读取

In [9]:
link_info = pd.read_csv(
    os.path.join(config.data_dir, "gy_contest_link_info.txt"), 
    delimiter = ";", 
    dtype = {"link_ID": object}
)
link_top = pd.read_csv(
    os.path.join(config.data_dir, "gy_contest_link_top.txt"), 
    delimiter = ";", 
    dtype = {"link_ID": object}
)
df = pd.read_csv(
    os.path.join(config.data_dir, "quaterfinal_gy_cmp_training_traveltime.txt"),
    delimiter = ";", 
    dtype = {"link_ID": object}
)

In [15]:
data_logging.data_logging(data = link_info, data_name = "link_info")

link_info.head()
               link_ID  length  width  link_class
0  4377906289869500514      57      3           1
1  4377906284594800514     247      9           1
2  4377906289425800514     194      3           1
3  4377906284525800514     839      3           1
4  4377906284422600514      55     12           1
link_info.tail()
                 link_ID  length  width  link_class
127  4377906288421600514     128      9           1
128  4377906289041600514     208      9           1
129  4377906286032600514     257     12           1
130  4377906281234600514      16      3           1
131  4377906286334600514      33     15           1
link_info.info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   link_ID     132 non-null    object
 1   length      132 non-null    int64 
 2   width       132 non-null    int64 
 3   link_class  132 non-null    

In [16]:
data_logging.data_logging(data = link_info, data_name = "link_top")

link_top.head()
               link_ID  length  width  link_class
0  4377906289869500514      57      3           1
1  4377906284594800514     247      9           1
2  4377906289425800514     194      3           1
3  4377906284525800514     839      3           1
4  4377906284422600514      55     12           1
link_top.tail()
                 link_ID  length  width  link_class
127  4377906288421600514     128      9           1
128  4377906289041600514     208      9           1
129  4377906286032600514     257     12           1
130  4377906281234600514      16      3           1
131  4377906286334600514      33     15           1
link_top.info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   link_ID     132 non-null    object
 1   length      132 non-null    int64 
 2   width       132 non-null    int64 
 3   link_class  132 non-null    int

In [14]:
data_logging.data_logging(data = df, data_name = "df")

df.head()
               link_ID        date                              time_interval  \
0  4377906283422600514  2017-05-06  [2017-05-06 11:04:00,2017-05-06 11:06:00)   
1  3377906289434510514  2017-05-06  [2017-05-06 10:42:00,2017-05-06 10:44:00)   
2  3377906285934510514  2017-05-06  [2017-05-06 11:56:00,2017-05-06 11:58:00)   
3  3377906285934510514  2017-05-06  [2017-05-06 17:46:00,2017-05-06 17:48:00)   
4  3377906287934510514  2017-05-06  [2017-05-06 10:52:00,2017-05-06 10:54:00)   

   travel_time  
0     1.386294  
1     0.693147  
2     3.589059  
3     3.303217  
4     2.433613  
df.tail()
                      link_ID        date  \
10632039  4377906287663800514  2017-07-31   
10632040  4377906288663800514  2017-07-31   
10632041  4377906288663800514  2017-07-31   
10632042  4377906288663800514  2017-07-31   
10632043  3377906289044510514  2017-07-31   

                                      time_interval  travel_time  
10632039  [2017-07-31 14:16:00,2017-07-31 14:18:00)  

## 2.探索性数据分析(EDA)

# 3.特征工程

In [10]:
# 特征变换
df["travel_time"] = np.log1p(df["travel_time"])

In [18]:
# 数据平滑
def quantile_clip(group):
#     group.plot()
    group[group < group.quantile(0.05)] = group.quantile(0.05)
    group[group > group.quantile(0.95)] = group.quantile(0.95)
#     group.plot()
#     plt.show()
    return group
df["travel_time"] = df.groupby(["link_ID", "date"])["travel_time"].transform(quantile_clip)

KeyboardInterrupt: 

In [ ]:
# 缺失值补全
date_range = pd.date_range("2016-07-01 00:00:00", "2016-07-31 00:00:00", freq = "2min") \
    .append(pd.date_range("2017-04-01 00:00:00", "2016-07-31 00:00:00", freq = "2min"))
new_index = pd.MultiIndex.from_product(
    [link_df["link_ID"].unique(), date_range], 
    names = ["link_ID", "time_interval_begin"]
)
df1 = pd.DateFrame(index = new_index).reset_index()
df3 = pd.merge(df1, df, on = ["link_ID", "time_interval_begin"], how = "left")